In [15]:
import numpy as np
import pandas as pd

from sklearn.metrics import (
    log_loss,
    roc_auc_score,
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
import optuna

def make_walk_forward_splits(
    data,
    train_years=2,
    val_months=1,
    test_months=2,
    step_months=2,
    use_val=True
):
    start = data.index.min() + pd.DateOffset(years=train_years)
    end = data.index.max()

    splits = []
    split_start = start

    while True:
        train_start = split_start - pd.DateOffset(years=train_years)
        train_end = split_start

        if use_val:
            val_start = train_end
            val_end = val_start + pd.DateOffset(months=val_months)
            test_start = val_end
            test_end = test_start + pd.DateOffset(months=test_months)
        else:
            val_start = None
            val_end = None
            test_start = train_end
            test_end = test_start + pd.DateOffset(months=test_months)

        if test_end > end:
            break

        train_idx = (data.index >= train_start) & (data.index < train_end)
        test_idx = (data.index >= test_start) & (data.index < test_end)

        val_idx = (data.index >= val_start) & (data.index < val_end) if use_val else None

        splits.append({
            "train_idx": train_idx,
            "val_idx": val_idx,
            "test_idx": test_idx,
            "train_start": train_start,
            "train_end": train_end,
            "val_start": val_start,
            "val_end": val_end,
            "test_start": test_start,
            "test_end": test_end,
        })

        split_start = split_start + pd.DateOffset(months=step_months)

    return splits

In [9]:
def classification_metrics(y_true, y_prob, threshold=0.5, eps=1e-12):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)
    y_prob = np.clip(y_prob, eps, 1 - eps)

    y_pred = (y_prob >= threshold).astype(int)

    out = {
        "logloss": log_loss(y_true, y_prob),
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "mean_prob": y_prob.mean(),
        "positive_rate_true": y_true.mean(),
        "positive_rate_pred": y_pred.mean(),
    }

    if len(np.unique(y_true)) == 2:
        out["auc"] = roc_auc_score(y_true, y_prob)
    else:
        out["auc"] = np.nan

    return out

In [18]:
def run_walk_forward_with_val_clf(
    model_fn,
    dataset,
    feature_cols,
    target_col="target_bin",
    output_path=None,
    threshold=0.5,
):
    splits = make_walk_forward_splits(
        dataset,
        train_years=2,
        val_months=1,
        test_months=2,
        step_months=2,
        use_val=True,
    )

    preds = []
    metrics = []

    for i, split in enumerate(splits):
        train = dataset.loc[split["train_idx"]].copy()
        test = dataset.loc[split["test_idx"]].copy()

        if train[target_col].nunique() < 2 or test[target_col].nunique() < 2:
            continue

        model = model_fn()
        model.fit(train[feature_cols], train[target_col])

        y_prob = model.predict_proba(test[feature_cols])[:, 1]
        y_true = test[target_col].astype(int).values
        y_pred = (y_prob >= threshold).astype(int)

        fold_preds = pd.DataFrame({
            "datetime": test.index,
            "fold": i,
            "y_true": y_true,
            "y_prob": y_prob,
            "y_pred": y_pred,
        })

        preds.append(fold_preds)

        fold_metrics = classification_metrics(y_true, y_prob, threshold=threshold)
        fold_metrics["fold"] = i
        fold_metrics["test_start"] = split["test_start"]
        fold_metrics["test_end"] = split["test_end"]
        metrics.append(fold_metrics)

    preds = pd.concat(preds).set_index("datetime")
    metrics = pd.DataFrame(metrics)

    if output_path is not None:
        preds.to_parquet(output_path)

    avg_metrics = metrics.drop(columns=["fold", "test_start", "test_end"]).mean()

    return preds, metrics, avg_metrics

In [5]:
df1h = pd.read_parquet('BTC_1h_OHLCV.parquet')
df6h = pd.read_parquet('BTC_6h_OHLCV.parquet')
df24h = pd.read_parquet('BTC_1d_OHLCV.parquet')

In [6]:
df1h.head()

,datetime,open,high,low,close,volume,turnover
0,2020-03-25 10:00:00+00:00,6500.0,6591.5,6500.0,6591.5,0.004,2.617950e+01
1,2020-03-25 11:00:00+00:00,6591.5,6628.5,6457.5,6511.5,438.873,2.877880e+06
2,2020-03-25 12:00:00+00:00,6511.5,6588.5,6502.0,6583.5,529.318,3.463795e+06
3,2020-03-25 13:00:00+00:00,6583.5,6745.5,6562.0,6585.0,449.162,2.971877e+06
4,2020-03-25 14:00:00+00:00,6585.0,6640.0,6516.0,6590.0,258.831,1.703331e+06


In [12]:
def make_features(df, horizon=1):
    df = df.copy()

    df["datetime"] = pd.to_datetime(df["datetime"])
    df = df.sort_values("datetime").reset_index(drop=True)

    df["log_close"] = np.log(df["close"])
    df["ret"] = df["log_close"].diff()

    df["future_ret"] = (
        df["log_close"].shift(-horizon) - df["log_close"]
    )

    df["target_bin"] = (df["future_ret"] > 0).astype(int)

    for lag in [1, 2, 3, 6, 12, 24]:
        df[f"ret_lag_{lag}"] = df["ret"].shift(lag)

    for w in [6, 12, 24, 72]:
        df[f"mom_{w}"] = df["ret"].rolling(w).sum().shift(1)
        df[f"vol_{w}"] = df["ret"].rolling(w).std().shift(1)

        df[f"vol_z_{w}"] = (
            (
                np.log1p(df["volume"])
                - np.log1p(df["volume"]).rolling(w).mean()
            )
            / (
                np.log1p(df["volume"]).rolling(w).std()
                + 1e-12
            )
        ).shift(1)

    df["hl"] = np.log(df["high"] / df["low"]).shift(1)

    df["oc"] = np.log(df["close"] / df["open"]).shift(1)

    df["close_pos"] = (
        (
            df["close"] - df["low"]
        )
        / (
            df["high"] - df["low"] + 1e-12
        )
    ).shift(1)

    feature_cols = [
        c for c in df.columns
        if c not in [
            "datetime",
            "open",
            "high",
            "low",
            "close",
            "volume",
            "turnover",
            "log_close",
            "future_ret",
            "target_bin",
        ]
    ]

    df = (
        df.replace([np.inf, -np.inf], np.nan)
          .dropna(subset=feature_cols + ["target_bin"])
          .reset_index(drop=True)
    )

    return df.set_index("datetime"), feature_cols

In [13]:
data1h, features1h = make_features(df1h, horizon=1)
data6h, features6h = make_features(df6h, horizon=1)
data24h, features24h = make_features(df24h, horizon=1)

In [21]:
def make_logreg_model(C):
    return make_pipeline(
        StandardScaler(),
        LogisticRegression(
            C=C,
            penalty="l2",
            solver="lbfgs",
            max_iter=2000,
            class_weight="balanced",
            random_state=42,
        )
    )

def tune_logreg(dataset, feature_cols, n_trials=30):
    splits = make_walk_forward_splits(
        dataset,
        train_years=2,
        val_months=1,
        test_months=2,
        step_months=2,
        use_val=True,
    )

    def objective(trial):
        C = trial.suggest_float("C", 1e-4, 100.0, log=True)
        losses = []

        for split in splits:
            train = dataset.loc[split["train_idx"]].copy()
            val = dataset.loc[split["val_idx"]].copy()

            if train["target_bin"].nunique() < 2 or val["target_bin"].nunique() < 2:
                continue

            model = make_logreg_model(C)
            model.fit(train[feature_cols], train["target_bin"])

            val_prob = model.predict_proba(val[feature_cols])[:, 1]
            losses.append(log_loss(val["target_bin"], val_prob))

        return np.mean(losses)

    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=n_trials)

    return study.best_params, study.best_value


def fit_best_logreg_oos(name, dataset, feature_cols, n_trials=30):
    best_params, best_score = tune_logreg(
        dataset=dataset,
        feature_cols=feature_cols,
        n_trials=n_trials,
    )

    def model_fn():
        return make_logreg_model(best_params["C"])

    preds, metrics, avg = run_walk_forward_with_val_clf(
        model_fn=model_fn,
        dataset=dataset,
        feature_cols=feature_cols,
        target_col="target_bin",
        output_path=f"{name}_logreg_preds.parquet",
        threshold=0.5,
    )

    preds = preds.rename(columns={
        "y_true": "target_sign",
        "y_prob": "pred_prob",
        "y_pred": "pred_sign",
    })

    preds["model"] = name
    preds["best_C"] = best_params["C"]
    preds["best_val_logloss"] = best_score

    return preds, metrics, avg, best_params

In [22]:
preds1h, metrics1h, avg1h, best1h = fit_best_logreg_oos(
    name="BTC_1h",
    dataset=data1h,
    feature_cols=features1h,
    n_trials=30,
)

preds6h, metrics6h, avg6h, best6h = fit_best_logreg_oos(
    name="BTC_6h",
    dataset=data6h,
    feature_cols=features6h,
    n_trials=30,
)

preds24h, metrics24h, avg24h, best24h = fit_best_logreg_oos(
    name="BTC_1d",
    dataset=data24h,
    feature_cols=features24h,
    n_trials=30,
)

[I 2026-05-17 17:43:26,643] A new study created in memory with name: no-name-3ca34979-c953-4587-aa53-6fe9436fc07b
[I 2026-05-17 17:43:27,054] Trial 0 finished with value: 0.6911104831137123 and parameters: {'C': 0.03165235832041182}. Best is trial 0 with value: 0.6911104831137123.
[I 2026-05-17 17:43:27,501] Trial 1 finished with value: 0.6911391380967826 and parameters: {'C': 3.363754348413407}. Best is trial 0 with value: 0.6911104831137123.
[I 2026-05-17 17:43:27,882] Trial 2 finished with value: 0.6911391670143407 and parameters: {'C': 3.7303473852473905}. Best is trial 0 with value: 0.6911104831137123.
[I 2026-05-17 17:43:28,269] Trial 3 finished with value: 0.691139160145899 and parameters: {'C': 3.636242105044557}. Best is trial 0 with value: 0.6911104831137123.
[I 2026-05-17 17:43:28,594] Trial 4 finished with value: 0.691068708508843 and parameters: {'C': 0.005942869130099029}. Best is trial 4 with value: 0.691068708508843.
[I 2026-05-17 17:43:28,985] Trial 5 finished with val